In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
df_all_friday = pd.read_csv('Friday_Botnet.csv')
df_all_friday.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,3268,112740690,32,16,6448,1152,403,0,201.5,204.724205,...,32,3.594286e+02,1.199802e+01,380,343,16100000.0,4.988048e+05,16400000,15400000,BENIGN
1,389,112740560,32,16,6448,5056,403,0,201.5,204.724205,...,32,3.202857e+02,1.574499e+01,330,285,16100000.0,4.987937e+05,16400000,15400000,BENIGN
2,0,113757377,545,0,0,0,0,0,0.0,0.000000,...,0,9.361829e+06,7.324646e+06,18900000,19,12200000.0,6.935824e+06,20800000,5504997,BENIGN
3,5355,100126,22,0,616,0,28,28,28.0,0.000000,...,32,0.000000e+00,0.000000e+00,0,0,0.0,0.000000e+00,0,0,BENIGN
4,0,54760,4,0,0,0,0,0,0.0,0.000000,...,0,0.000000e+00,0.000000e+00,0,0,0.0,0.000000e+00,0,0,BENIGN


In [3]:
df_botnet = df_all_friday[df_all_friday[' Label'].str.lower().str.contains("bot", na=False)]
df_benign = df_all_friday[df_all_friday[' Label'].str.lower().str.contains("benign", na=False)]

In [4]:
df_home = pd.read_csv('home_flows.csv')
df_home_us = df_home.sample(n=2000, random_state=42, replace=False)

In [5]:
df_benign_us = df_benign.sample(n=6000, random_state=42, replace=False)

In [6]:
df_nonbot = pd.concat([df_benign_us, df_home_us], ignore_index=True)

In [7]:
CICIDS_RENAME = {
    'Destination Port':      'dsport',
    'Flow Duration':         'dur',
    'Total Fwd Packets':     'Spkts',
    'Total Backward Packets':'Dpkts',
    'Total Length of Fwd Packets': 'sbytes',
    'Total Length of Bwd Packets': 'dbytes',
    'Avg Fwd Segment Size':  'smeansz',
    'Avg Bwd Segment Size':  'dmeansz',
    'Flow Bytes/s':          'Sload',
    'Fwd IAT Mean':          'Sintpkt',
    'Bwd IAT Mean':          'Dintpkt',
    'Flow IAT Std':          'Sjit',
    'Bwd IAT Std':           'Djit',
    'Flow IAT Min':          'tcprtt',
    'Label':                 'label',
}

In [8]:
import ipaddress
def ip_flags(ip_series):
    """Returns (is_private, is_loopback) Series for a column of IP strings."""
    is_private  = []
    is_loopback = []
    for ip in ip_series:
        try:
            addr = ipaddress.ip_address(str(ip).strip())
            is_private.append(int(addr.is_private))
            is_loopback.append(int(addr.is_loopback))
        except ValueError:
            is_private.append(0)
            is_loopback.append(0)
    return pd.Series(is_private, dtype=int), pd.Series(is_loopback, dtype=int)

In [9]:
df_fri = pd.read_csv("Friday_Botnet.csv")
df_fri.columns = df_fri.columns.str.strip()
df_fri = df_fri.rename(columns=CICIDS_RENAME)

In [10]:
df_botnet = df_fri[df_fri['label'].str.lower().str.contains("bot", na=False)]
df_benign = df_fri[df_fri['label'].str.lower().str.contains("benign", na=False)]

In [11]:
df_bot = df_botnet[['dsport', 'dur', 'sbytes', 'dbytes','Spkts', 'Dpkts', 'smeansz', 'dmeansz', 'Sload','Sintpkt', 'Dintpkt', 'Sjit', 'Djit', 'tcprtt']].copy()


In [12]:
df_bot['pkt_ratio'] = df_bot['Spkts']  / (df_bot['Dpkts']  + 1e-9)
df_bot['byte_ratio']= df_bot['sbytes'] / (df_bot['dbytes'] + 1e-9)
df_bot['src_is_private'] = 0        # no raw IPs in CIC-IDS
df_bot['dst_is_private'] = 0
df_bot['src_is_loopback'] = 0
df_bot['dst_is_loopback'] = 0
df_bot['is_sm_ips_ports'] = 0
df_bot['target'] = 1   

In [13]:
df_ben_fri = df_benign[['dsport','dur','sbytes','dbytes','Spkts','Dpkts','smeansz','dmeansz','Sload','Sintpkt','Dintpkt','Sjit','Djit','tcprtt']].copy()

In [14]:
df_ben_fri['pkt_ratio'] = df_ben_fri['Spkts']  / (df_ben_fri['Dpkts']  + 1e-9)
df_ben_fri['byte_ratio'] = df_ben_fri['sbytes'] / (df_ben_fri['dbytes'] + 1e-9)
df_ben_fri['src_is_private'] = 0
df_ben_fri['dst_is_private'] = 0
df_ben_fri['src_is_loopback'] = 0
df_ben_fri['dst_is_loopback'] = 0
df_ben_fri['is_sm_ips_ports'] = 0
df_ben_fri['target'] = 0

In [15]:
df_ben_fri = df_ben_fri.sample(n=6000, random_state=42, replace=False)

In [16]:
df_home = pd.read_csv('home_flows.csv')
df_home_us = df_home.sample(n=2000, random_state=42, replace=False).copy()

In [17]:
src_priv, src_loop = ip_flags(df_home_us['srcip'])
dst_priv, dst_loop = ip_flags(df_home_us['dstip'])

In [18]:
df_home_feat = pd.DataFrame({
    'dsport':           df_home_us['dsport'].values,
    'dur':              df_home_us['dur'].values,
    'sbytes':           df_home_us['sbytes'].values,
    'dbytes':           df_home_us['dbytes'].values,
    'Spkts':            df_home_us['Spkts'].values,
    'Dpkts':            df_home_us['Dpkts'].values,
    'smeansz':          df_home_us['smeansz'].values,
    'dmeansz':          df_home_us['dmeansz'].values,
    'Sload':            df_home_us['Sload'].values,
    'pkt_ratio':        df_home_us['Spkts'].values / (df_home_us['Dpkts'].values + 1e-9),
    'byte_ratio':       df_home_us['sbytes'].values / (df_home_us['dbytes'].values + 1e-9),
    'Sintpkt':          df_home_us['Sintpkt'].values,
    'Dintpkt':          df_home_us['Dintpkt'].values,
    'Sjit':             df_home_us['Sjit'].values,
    'Djit':             df_home_us['Djit'].values,
    'tcprtt':           df_home_us['tcprtt'].values,
    'src_is_private':   src_priv.values,
    'dst_is_private':   dst_priv.values,
    'src_is_loopback':  src_loop.values,
    'dst_is_loopback':  dst_loop.values,
    'is_sm_ips_ports':  df_home_us['is_sm_ips_ports'].values,
    'target':           0,           # all home flows are benign = 0
})

In [19]:
FEATURES = [
    'dsport', 'dur', 
    'smeansz', 'dmeansz', 'Sload', 'pkt_ratio', 'byte_ratio',
    'Sintpkt', 'Dintpkt', 'Sjit', 'Djit', 'tcprtt', 'is_sm_ips_ports',
    'src_is_private', 'dst_is_private', 'src_is_loopback', 'dst_is_loopback',
]

df_all = pd.concat([df_bot[FEATURES + ['target']],df_ben_fri[FEATURES + ['target']],df_home_feat[FEATURES + ['target']],], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

In [20]:
df_all.replace([np.inf, -np.inf], np.nan, inplace=True)
df_all.fillna(0, inplace=True)

,dsport,dur,smeansz,dmeansz,Sload,pkt_ratio,byte_ratio,Sintpkt,Dintpkt,Sjit,Djit,tcprtt,is_sm_ips_ports,src_is_private,dst_is_private,src_is_loopback,dst_is_loopback,target
0,53,4.745500e+04,45.00,356.000000,8.450111e+03,1.000000e+00,1.264045e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,47455.0,0,0,0,0,0,0
1,52089,1.000000e+01,140.00,0.000000,2.800000e+07,2.000000e+09,2.800000e+11,1.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,10.0,0,0,0,0,0,0
2,2960,2.000000e+01,6.00,6.000000,6.000000e+05,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,20.0,0,0,0,0,0,1
3,8080,1.029548e+06,0.00,6.000000,1.748340e+01,1.000000e+00,0.000000e+00,5.145010e+05,5.143845e+05,2.811154e+05,1.832114e+03,514.0,0,0,0,0,0,1
4,52195,1.140839e+01,55.20,55.200000,3.870835e+02,1.000000e+00,1.000000e+00,1.267599e+00,1.267599e+00,9.404622e-01,9.404622e-01,0.0,0,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9961,443,1.194713e+08,119.15,243.545455,6.479380e+01,9.090909e-01,4.447555e-01,6.287964e+06,5.684670e+06,1.270000e+07,1.750000e+07,1.0,0,0,0,0,0,0
9962,53,3.027100e+04,55.00,183.000000,7.862310e+03,1.000000e+00,3.005464e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,30271.0,0,0,0,0,0,0
9963,53,1.750000e+02,37.00,94.000000,1.497143e+06,1.000000e+00,3.936170e-01,4.900000e+01,3.000000e+00,6.054200e+01,0.000000e+00,3.0,0,0,0,0,0,0
9964,8080,8.442500e+04,51.75,44.666667,4.039088e+03,1.333333e+00,1.544776e+00,2.814167e+04,4.144550e+04,3.339286e+04,5.767799e+04,13.0,0,0,0,0,0,1


In [21]:
X = df_all[FEATURES].values
y = df_all['target'].values

In [22]:
print(f"  Botnet  (label=1) : {(y==1).sum()}")
print(f"  Benign  (label=0) : {(y==0).sum()}")
print(f"  Total             : {len(y)}")

  Botnet  (label=1) : 1966
  Benign  (label=0) : 8000
  Total             : 9966


In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

In [24]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

In [25]:
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_feat

In [26]:
y_pred = rf.predict(X_test)

In [27]:
print(classification_report(y_test, y_pred,target_names=['Benign', 'Botnet'],zero_division=0))

              precision    recall  f1-score   support

      Benign       1.00      0.99      0.99      1601
      Botnet       0.96      0.99      0.98       393

    accuracy                           0.99      1994
   macro avg       0.98      0.99      0.99      1994
weighted avg       0.99      0.99      0.99      1994



In [28]:
cm = confusion_matrix(y_test, y_pred)
print(f"              Predicted Benign  Predicted Botnet")
print(f"Actual Benign      {cm[0][0]:<10}   {cm[0][1]}")
print(f"Actual Botnet      {cm[1][0]:<10}   {cm[1][1]}")

              Predicted Benign  Predicted Botnet
Actual Benign      1585         16
Actual Botnet      3            390


In [29]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
cv_scores.mean()*100

np.float64(98.60527895779663)

In [30]:
import pickle


with open("botnet_rf_model.pkl", "wb") as f:
    pickle.dump(rf, f)